# آماده‌سازی داده MovieLens برای پروژه پیشنهاد فیلم

این نوت‌بوک دیتاست رسمی `ml-latest-small` را دانلود می‌کند، داده‌ها را پاک‌سازی می‌کند و با جداسازی زمانیِ جداگانه برای هر کاربر، مجموعه آموزش و آزمون می‌سازد.

**نکته:** پوشه داده در `.gitignore` قرار دارد و نباید بدون بررسی مجوز، فایل‌های خام را در مخزن عمومی قرار دهید.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import json

# این نوت‌بوک را از ریشه پوشه step2_movielens اجرا کنید.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists() and (PROJECT_ROOT.parent / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'scripts' / 'prepare_movielens.py').exists():
    raise FileNotFoundError('پوشه scripts پیدا نشد؛ نوت‌بوک را از ریشه پروژه اجرا کنید.')
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from prepare_movielens import prepare_movielens


In [ ]:
summary = prepare_movielens(PROJECT_ROOT)
pd.Series(summary, name='value').to_frame()


## مشاهده خروجی‌های پاک‌سازی‌شده
در این مرحله، `train.csv` و `test.csv` هر دو شامل شناسه کاربر، شناسه فیلم، امتیاز، زمان، عنوان و ژانر هستند.


In [ ]:
processed = PROJECT_ROOT / 'data' / 'processed'
train = pd.read_csv(processed / 'train.csv')
test = pd.read_csv(processed / 'test.csv')
movies = pd.read_csv(processed / 'movies_clean.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Movies shape:', movies.shape)
display(train.head())
display(test.head())


In [ ]:
print('Missing values in train:')
display(train.isna().sum())
print('Rating distribution in train:')
display(train['rating'].value_counts().sort_index())
print('Unique users in train/test:', train['userId'].nunique(), test['userId'].nunique())
print('Test rows:', len(test))


## تفسیر روش جداسازی
برای هر کاربر، امتیازها بر اساس `timestamp` مرتب شده‌اند. ۸۰ درصد تعاملات قدیمی‌تر وارد آموزش و ۲۰ درصد جدیدتر وارد آزمون می‌شوند. این روش برای شبیه‌سازی پیش‌بینی تعاملات آینده مناسب‌تر از مخلوط‌کردن تصادفی گذشته و آینده است. در مرحله مدل‌سازی باید پیشنهادها فقط با داده آموزش ساخته شوند و فیلم‌هایی که کاربر قبلاً در آموزش امتیاز داده است از فهرست پیشنهاد حذف شوند.
